# 07 AKShare 数据获取 + SQLite 持久化

## 这一讲做什么？

之前我们在 Notebook 05 里用 AKShare 下载了沪深300，但每次都要重新下载——慢，而且浪费网络资源。
这一讲教你**把数据存到本地数据库**，下次直接用，不用重新下载。

### SQLite 是什么？

SQLite 是一个「文件型数据库」——整个数据库就是一个 `.db` 文件，不需要安装服务器。
它比 CSV 文件好在：可以像 Excel 筛选一样用 SQL 查询，而且速度快得多。

### SQL 极简入门（15 秒版）

SQL 只有四种基本操作，记住缩写 **CRUD**：

| 操作 | SQL | 含义 |
|------|-----|------|
| **C**reate | `INSERT INTO ... VALUES (...)` | 插入新数据 |
| **R**ead | `SELECT ... FROM ... WHERE ...` | 查询数据 |
| **U**pdate | `UPDATE ... SET ... WHERE ...` | 修改数据 |
| **D**elete | `DELETE FROM ... WHERE ...` | 删除数据 |

这一讲主要用 **Create**（存入）和 **Read**（取出）。

**学习目标**
- 从 AKShare 获取至少 5 种 A 股数据
- 数据持久化到本地 SQLite 数据库
- 用 SQL 查询已存储的数据
- 建立可复用的数据获取管道

---


In [1]:
# ==================== 导入 & 配置 ====================
import akshare as ak
import pandas as pd
import sqlite3
import os
from datetime import datetime, timedelta
from contextlib import contextmanager
import warnings
warnings.filterwarnings('ignore')

# 数据库路径（可自定义）
DB_PATH = os.path.join(os.path.dirname(os.path.abspath('.')), 'data', 'a_stock.db')
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

print(f'✅ 环境就绪  |  AKShare v{ak.__version__}  |  数据库: {DB_PATH}')

✅ 环境就绪  |  AKShare v1.18.64  |  数据库: /Users/echo/Desktop/python/data/a_stock.db


---
## 一、SQLite 数据库工具函数

封装数据库操作的通用函数，方便后续所有数据写入统一调用。

In [ ]:
# ─── SQL 查询 ───
# SQL 语句是大写（习惯，不是必须），表名和列名是小写
# 每一句 SQL 以分号 ; 结尾
# 
# SELECT 列名 FROM 表名 WHERE 条件;
#    ↑ 想要什么  ↑ 从哪取    ↑ 过滤条件（可选）
# ==================== SQLite 工具函数库 ====================

@contextmanager
def get_db(db_path=DB_PATH):
    """获取数据库连接（上下文管理器，自动关闭）"""
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def save_to_sqlite(df, table_name, db_path=DB_PATH, if_exists='replace', index=False):
    """将 DataFrame 保存到 SQLite 表
    
    Parameters
    ----------
    df : pd.DataFrame     数据
    table_name : str      表名
    db_path : str         数据库路径
    if_exists : str       'replace' 覆盖 / 'append' 追加
    index : bool          是否保存索引
    
    Returns
    -------
    int  写入行数
    """
    with get_db(db_path) as conn:
        df.to_sql(table_name, conn, if_exists=if_exists, index=index)
    return len(df)


def read_from_sqlite(table_name, db_path=DB_PATH, condition=None, limit=None):
    """从 SQLite 读取数据
    
    Parameters
    ----------
    table_name : str      表名
    db_path : str         数据库路径
    condition : str       可选 WHERE 条件，如 "date >= '2026-01-01'"
    limit : int           可选返回行数
    
    Returns
    -------
    pd.DataFrame
    """
    query = f"SELECT * FROM {table_name}"
    if condition:
        query += f" WHERE {condition}"
    if limit:
        query += f" LIMIT {limit}"
    with get_db(db_path) as conn:
        return pd.read_sql_query(query, conn)


def get_table_info(table_name, db_path=DB_PATH):
    """查看表结构（列名 + 数据类型）"""
    with get_db(db_path) as conn:
        cur = conn.execute(f"PRAGMA table_info({table_name})")
        return pd.DataFrame(cur.fetchall(), columns=['cid','name','type','notnull','dflt','pk'])


def list_tables(db_path=DB_PATH):
    """列出数据库中所有表"""
    with get_db(db_path) as conn:
        cur = conn.execute("SELECT name FROM sqlite_master WHERE type='table'")
        return [row['name'] for row in cur.fetchall()]


def get_row_count(table_name, db_path=DB_PATH):
    """获取表行数"""
    with get_db(db_path) as conn:
        cur = conn.execute(f"SELECT COUNT(*) as cnt FROM {table_name}")
        return cur.fetchone()['cnt']


print('✅ SQLite 工具函数已定义')
print(f'   get_db()          → 上下文管理器连接')
print(f'   save_to_sqlite()  → DataFrame 写入')
print(f'   read_from_sqlite()→ 查询读取')
print(f'   get_table_info()  → 表结构查看')
print(f'   list_tables()     → 列出所有表')
print(f'   get_row_count()   → 行数统计')

---
## 二、数据获取函数封装

每个函数封装一个 AKShare 数据源，返回标准化的 DataFrame。

In [ ]:
# ==================== 1. 个股日线行情 ====================

def get_stock_daily(symbol, start_date, end_date, adjust='qfq'):
    """获取 A 股个股日线行情
    
    Parameters
    ----------
    symbol : str       股票代码，如 '000001'（平安银行）
    start_date : str   起始日期 'YYYYMMDD'
    end_date : str     结束日期 'YYYYMMDD'
    adjust : str       复权方式 'qfq'前复权 / 'hfq'后复权 / ''不复权
    
    Returns
    -------
    pd.DataFrame
    """
    df = ak.stock_zh_a_hist(
        symbol=symbol, period='daily',
        start_date=start_date, end_date=end_date, adjust=adjust
    )
    df['symbol'] = symbol
    # 统一列名（中→英）
    df = df.rename(columns={
        '日期': 'date', '开盘': 'open', '收盘': 'close',
        '最高': 'high', '最低': 'low', '成交量': 'volume',
        '成交额': 'amount', '振幅': 'amplitude', '涨跌幅': 'pct_change',
        '涨跌额': 'change', '换手率': 'turnover'
    })
    df['date'] = pd.to_datetime(df['date'])
    return df


# 测试
stock_000001 = get_stock_daily('000001', '20260101', '20260531')
print(f'✅ 个股日线 | 平安银行(000001) | {len(stock_000001)} 条记录')
stock_000001.head(3)

In [ ]:
# ==================== 2. 指数日线行情 ====================

def get_index_daily(symbol, start_date, end_date):
    """获取指数日线行情
    
    Parameters
    ----------
    symbol : str       指数代码
                       'sh000001' 上证综指, 'sz399001' 深证成指,
                       'sh000300' 沪深300, 'sz399006' 创业板指
    start_date : str   起始日期 'YYYY-MM-DD' 或 'YYYYMMDD'
    end_date : str     结束日期
    
    Returns
    -------
    pd.DataFrame
    """
    df = ak.stock_zh_index_daily_em(symbol=symbol, start_date=start_date, end_date=end_date)
    df['symbol'] = symbol
    df = df.rename(columns={
        'date': 'date', 'open': 'open', 'close': 'close',
        'high': 'high', 'low': 'low', 'volume': 'volume', 'amount': 'amount'
    })
    df['date'] = pd.to_datetime(df['date'])
    return df


# 测试：沪深300
idx_hs300 = get_index_daily('sh000300', '2026-01-01', '2026-05-31')
print(f'✅ 指数日线 | 沪深300 | {len(idx_hs300)} 条记录')
idx_hs300.tail(3)

In [ ]:
# ==================== 3. 行业板块日线行情 ====================

def get_industry_board(symbol, start_date, end_date):
    """获取东方财富行业板块日线行情
    
    Parameters
    ----------
    symbol : str       板块名称，如 '半导体', '银行', '白酒', '新能源汽车'
    start_date : str   起始日期 'YYYYMMDD'
    end_date : str     结束日期
    
    Returns
    -------
    pd.DataFrame
    """
    df = ak.stock_board_industry_hist_em(
        symbol=symbol, period='daily',
        start_date=start_date, end_date=end_date, adjust=''
    )
    df['sector'] = symbol
    df = df.rename(columns={
        '日期': 'date', '开盘': 'open', '收盘': 'close',
        '最高': 'high', '最低': 'low', '成交量': 'volume',
        '成交额': 'amount', '涨跌幅': 'pct_change'
    })
    df['date'] = pd.to_datetime(df['date'])
    return df


# 测试
sector_semi = get_industry_board('半导体', '20260101', '20260531')
sector_bank = get_industry_board('银行', '20260101', '20260531')
print(f'✅ 行业板块 | 半导体 {len(sector_semi)} 条 | 银行 {len(sector_bank)} 条')
sector_semi.head(3)

In [ ]:
# ==================== 4. 个股资金流向 ====================

def get_stock_fund_flow(stock, market='sh'):
    """获取个股历史资金流向
    
    Parameters
    ----------
    stock : str    股票代码，如 '000001'
    market : str   市场 'sh'上海 / 'sz'深圳
    
    Returns
    -------
    pd.DataFrame
    """
    df = ak.stock_individual_fund_flow(stock=stock, market=market)
    df['symbol'] = f'{market}.{stock}'
    df = df.rename(columns={
        '日期': 'date', '收盘价': 'close', '涨跌幅': 'pct_change',
        '主力净流入-净额': 'main_net_inflow',
        '主力净流入-净占比': 'main_net_inflow_pct',
        '超大单净流入-净额': 'super_large_net_inflow',
        '超大单净流入-净占比': 'super_large_net_inflow_pct',
        '大单净流入-净额': 'large_net_inflow',
        '大单净流入-净占比': 'large_net_inflow_pct',
        '中单净流入-净额': 'mid_net_inflow',
        '中单净流入-净占比': 'mid_net_inflow_pct',
        '小单净流入-净额': 'small_net_inflow',
        '小单净流入-净占比': 'small_net_inflow_pct'
    })
    df['date'] = pd.to_datetime(df['date'])
    return df


# 测试
fund_000001 = get_stock_fund_flow('000001', 'sz')
print(f'✅ 资金流向 | 平安银行 | {len(fund_000001)} 条记录')
fund_000001.tail(3)

In [ ]:
# ==================== 5. 龙虎榜明细 ====================

def get_top_trader_board(date):
    """获取指定日期的龙虎榜明细
    
    Parameters
    ----------
    date : str  日期 'YYYYMMDD' 如 '20260529'
    
    Returns
    -------
    pd.DataFrame
    """
    df = ak.stock_sina_lhb_detail_daily(date=date)
    df['trade_date'] = date
    df = df.rename(columns={
        '序号': 'rank', '股票代码': 'symbol', '股票名称': 'name',
        '收盘价': 'close', '涨跌幅': 'pct_change',
        '龙虎榜净买额': 'lhb_net_buy', '龙虎榜买入额': 'lhb_buy',
        '龙虎榜卖出额': 'lhb_sell', '龙虎榜成交额': 'lhb_amount',
        '市场总成交额': 'market_amount', '净买额占总成交比': 'net_buy_pct',
        '成交额占总成交比': 'amount_pct', '换手率': 'turnover',
        '上榜原因': 'reason', '流通市值': 'float_mv',
        '解读': 'interpretation'
    })
    return df


# 测试：取最近交易日
recent_date = '20260529'
try:
    lhb = get_top_trader_board(recent_date)
    print(f'✅ 龙虎榜   | {recent_date} | {len(lhb)} 只上榜股票')
    lhb[['symbol','name','pct_change','lhb_net_buy','reason']].head(5)
except Exception as e:
    print(f'⚠️ 龙虎榜 {recent_date} 暂无数据（非交易日或数据未更新）')
    print(f'   错误信息: {e}')

In [ ]:
# ==================== 🎁 Bonus: 全市场实时快照 ====================

def get_market_spot():
    """获取 A 股全市场实时行情快照（市值、PE、涨跌幅等）
    
    注：数据量大（5000+ 只股票），建议日终调用一次
    
    Returns
    -------
    pd.DataFrame
    """
    df = ak.stock_zh_a_spot_em()
    df = df.rename(columns={
        '代码': 'symbol', '名称': 'name', '最新价': 'price',
        '涨跌幅': 'pct_change', '涨跌额': 'change',
        '成交量': 'volume', '成交额': 'amount',
        '振幅': 'amplitude', '最高': 'high', '最低': 'low',
        '今开': 'open', '昨收': 'pre_close',
        '量比': 'volume_ratio', '换手率': 'turnover',
        '市盈率-动态': 'pe_dynamic', '市净率': 'pb'
    })
    df['update_time'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return df


# 测试
spot = get_market_spot()
print(f'✅ 全市场快照 | {len(spot)} 只股票')
print(f'   涨幅榜 Top 5:')
spot.nlargest(5, 'pct_change')[['symbol','name','price','pct_change']]

---
## 三、批量拉取 & 持久化

把所有数据写入 SQLite，每类数据一张表。

In [ ]:
# ==================== 批量拉取并写入 SQLite ====================

print('🚀 开始批量拉取数据...\n')

results = []

# 1. 个股日线（3只代表性股票）
stocks = {
    '000001': '平安银行',
    '600519': '贵州茅台',
    '300750': '宁德时代'
}
stock_dfs = []
for code, name in stocks.items():
    try:
        df = get_stock_daily(code, '20260101', '20260531')
        df['name'] = name
        stock_dfs.append(df)
        print(f'  ✓ {code} {name:<6} {len(df)} 条')
    except Exception as e:
        print(f'  ✗ {code} {name}: {e}')

all_stocks = pd.concat(stock_dfs, ignore_index=True)
n = save_to_sqlite(all_stocks, 'stock_daily', if_exists='replace')
results.append(('个股日线', 'stock_daily', n))

# 2. 指数日线
indices = {
    'sh000001': '上证综指',
    'sh000300': '沪深300',
    'sz399006': '创业板指'
}
idx_dfs = []
for code, name in indices.items():
    try:
        df = get_index_daily(code, '2026-01-01', '2026-05-31')
        df['name'] = name
        idx_dfs.append(df)
        print(f'  ✓ {code:<10} {name:<6} {len(df)} 条')
    except Exception as e:
        print(f'  ✗ {code} {name}: {e}')

all_indices = pd.concat(idx_dfs, ignore_index=True)
n = save_to_sqlite(all_indices, 'index_daily', if_exists='replace')
results.append(('指数日线', 'index_daily', n))

# 3. 行业板块（3个行业）
sectors = ['半导体', '银行', '白酒']
sec_dfs = []
for s in sectors:
    try:
        df = get_industry_board(s, '20260101', '20260531')
        sec_dfs.append(df)
        print(f'  ✓ {s:<8} {len(df)} 条')
    except Exception as e:
        print(f'  ✗ {s}: {e}')

all_sectors = pd.concat(sec_dfs, ignore_index=True)
n = save_to_sqlite(all_sectors, 'industry_board', if_exists='replace')
results.append(('行业板块', 'industry_board', n))

# 4. 资金流向
fund_stocks = [('000001', 'sz'), ('600519', 'sh'), ('300750', 'sz')]
fund_dfs = []
for code, mkt in fund_stocks:
    try:
        df = get_stock_fund_flow(code, mkt)
        fund_dfs.append(df)
        print(f'  ✓ {code} ({mkt})  {len(df)} 条')
    except Exception as e:
        print(f'  ✗ {code}: {e}')

all_funds = pd.concat(fund_dfs, ignore_index=True)
n = save_to_sqlite(all_funds, 'stock_fund_flow', if_exists='replace')
results.append(('资金流向', 'stock_fund_flow', n))

# 5. 龙虎榜
try:
    lhb = get_top_trader_board('20260529')
    n = save_to_sqlite(lhb, 'lhb_detail', if_exists='replace')
    results.append(('龙虎榜', 'lhb_detail', n))
    print(f'  ✓ 龙虎榜     {n} 条')
except Exception as e:
    results.append(('龙虎榜', 'lhb_detail', 0))
    print(f'  ⚠ 龙虎榜 暂无数据')

# 6. 全市场快照（Bonus）
try:
    spot = get_market_spot()
    n = save_to_sqlite(spot, 'market_spot', if_exists='replace')
    results.append(('全市场快照', 'market_spot', n))
    print(f'  ✓ 全市场     {n} 条')
except Exception as e:
    results.append(('全市场快照', 'market_spot', 0))
    print(f'  ⚠ 全市场快照 {e}')

print(f'\n📊 写入汇总:')
total = 0
for label, table, count in results:
    print(f'   {label:<10} → {table:<18} {count:>6} 行')
    total += count
print(f'   {"总计":<10}   {"":<18} {total:>6} 行')

---
## 四、数据验证

从 SQLite 反读数据，验证完整性。

In [ ]:
# ==================== 验证：读取 & 查询 ====================

print('📋 数据库总览\n')

tables = list_tables()
for t in tables:
    count = get_row_count(t)
    print(f'   {t:<22} {count:>7} 行')

print(f'\n--- 个股日线 表结构 ---')
display(get_table_info('stock_daily'))

print(f'\n--- 查询示例：贵州茅台 2026年5月 行情 ---')
moutai = read_from_sqlite(
    'stock_daily',
    condition="symbol='600519' AND date >= '2026-05-01'"
)
display(moutai[['date','symbol','open','close','high','low','pct_change']].tail(5))

print(f'\n--- 资金流向：主力净流入 Top 5 ---')
fund_query = read_from_sqlite('stock_fund_flow', limit=100)
if 'main_net_inflow' in fund_query.columns and not fund_query.empty:
    top_inflow = fund_query.nlargest(5, 'main_net_inflow')
    display(top_inflow[['date','symbol','close','main_net_inflow','main_net_inflow_pct']])

# ⚠ 注意：下面这个 import 只在你把 notebook 导出为 .py 文件后才有效
# 在 notebook 里直接运行会报错——因为 notebook 本身不是 .py 模块
# 解决方法：把需要复用的函数定义在一个独立的 .py 文件里
# from utils_sqlite import save_to_sqlite, read_from_sqlite, get_stock_daily

# 暂时注释掉上面的 import，直接在本 notebook 中定义函数
